# Phase A Validation Notebook

This notebook is the inspection and presentation surface for the Phase A NLP Refinement Layer. It either loads durable refined outputs or runs a selected refinement tier from the existing silver candidate dataset, then compares the MVP `vehicle_normalized` rankings with refined Vehicle Head lemma rankings.

Phase A is structural vehicle refinement. It does not classify candidates as figurative or literal.

In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "tal_qual").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/tal_qual")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT

In [ ]:
from pyspark.sql import SparkSession
import tempfile
import zipfile

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("tal-qual-phase-a-validation")
    .config("spark.default.parallelism", "1")
    .config("spark.sql.shuffle.partitions", "1")
    .getOrCreate()
)

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))
spark.version

In [ ]:
from time import perf_counter

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

from tal_qual import (
    ONE_SHARD_REFINED_TIER,
    PHASE_A_QUALITY_BUCKET_COUNTS_OUTPUT_PATH,
    PHASE_A_REFINEMENT_EXAMPLES_OUTPUT_PATH,
    PHASE_A_SCOPE_COUNTS_OUTPUT_PATH,
    PHASE_A_TOP_CHARTABLE_VEHICLE_HEADS_OUTPUT_PATH,
    PHASE_A_TOP_CLEAN_COMMON_NOUN_HEADS_OUTPUT_PATH,
    PHASE_A_TOP_VEHICLE_HEADS_BY_PATTERN_OUTPUT_PATH,
    REFINED_CANDIDATES_NLP_OUTPUT_PATH,
    SAMPLE_DEBUG_TIER,
    SILVER_OUTPUT_PATH,
    load_portuguese_parser,
    prepare_refined_dataframe_for_tier,
    refinement_examples_dataframe,
    refinement_scope_counts_dataframe,
    structural_quality_bucket_counts_dataframe,
    top_chartable_vehicle_heads_dataframe,
    top_clean_common_noun_vehicle_heads_dataframe,
    top_refined_vehicle_heads_by_pattern_dataframe,
    top_vehicles_global_dataframe,
    write_phase_a_csv_outputs,
    write_refined_parquet,
)

RUN_TIER = os.environ.get("TAL_QUAL_PHASE_A_TIER", SAMPLE_DEBUG_TIER)
ROWS_PER_PATTERN = int(os.environ.get("TAL_QUAL_PHASE_A_ROWS_PER_PATTERN", "25"))
LOAD_EXISTING_REFINED = os.environ.get("TAL_QUAL_PHASE_A_LOAD_EXISTING", "1") != "0"

silver_output_path = REPO_ROOT / SILVER_OUTPUT_PATH
canonical_refined_output_path = REPO_ROOT / REFINED_CANDIDATES_NLP_OUTPUT_PATH
sample_debug_refined_output_path = canonical_refined_output_path.with_name(
    f"{canonical_refined_output_path.name}_sample_debug"
)
refined_output_path = (
    sample_debug_refined_output_path if RUN_TIER == SAMPLE_DEBUG_TIER else canonical_refined_output_path
)

phase_a_csv_paths = {
    "scope_counts": REPO_ROOT / PHASE_A_SCOPE_COUNTS_OUTPUT_PATH,
    "quality_bucket_counts": REPO_ROOT / PHASE_A_QUALITY_BUCKET_COUNTS_OUTPUT_PATH,
    "top_clean_common_noun_heads": REPO_ROOT / PHASE_A_TOP_CLEAN_COMMON_NOUN_HEADS_OUTPUT_PATH,
    "top_chartable_vehicle_heads": REPO_ROOT / PHASE_A_TOP_CHARTABLE_VEHICLE_HEADS_OUTPUT_PATH,
    "top_vehicle_heads_by_pattern": REPO_ROOT / PHASE_A_TOP_VEHICLE_HEADS_BY_PATTERN_OUTPUT_PATH,
    "refinement_examples": REPO_ROOT / PHASE_A_REFINEMENT_EXAMPLES_OUTPUT_PATH,
}

assert RUN_TIER in {SAMPLE_DEBUG_TIER, ONE_SHARD_REFINED_TIER}, f"Unsupported tier: {RUN_TIER}"
assert silver_output_path.exists(), f"Missing silver candidates: {silver_output_path}"

parser_probe = load_portuguese_parser()
assert not getattr(parser_probe, "parser_unavailable", False), (
    "Missing Phase A spaCy parser. Install project dependencies in this runtime, "
    "for example: python -m pip install -e /home/jovyan/work"
)
phase_a_parser_model = f"{parser_probe.model_name} {parser_probe.model_version}".strip()

RUN_TIER, silver_output_path, refined_output_path, phase_a_parser_model

## Load Or Run Phase A

The notebook defaults to `sample_debug` so it can run quickly in the expected local Docker/Jupyter workflow. Set `TAL_QUAL_PHASE_A_TIER=one_shard_refined` before launching Jupyter to run the full one-shard refinement tier.

In [ ]:
from pyspark.sql.functions import col


def loaded_refined_output_matches_parser(dataframe, parser_model_version: str) -> bool:
    if not parser_model_version:
        return False

    target_model_versions = {
        row["nlp_model_version"]
        for row in (
            dataframe
            .where(col("nlp_refinement_scope").isin("primary_nominal_article", "primary_nominal_bare"))
            .select("nlp_model_version")
            .distinct()
            .collect()
        )
    }
    return parser_model_version in target_model_versions


def run_refinement():
    refined_source_df = prepare_refined_dataframe_for_tier(
        silver_df,
        tier=RUN_TIER,
        rows_per_pattern=ROWS_PER_PATTERN,
    )
    write_refined_parquet(refined_source_df, refined_output_path)
    return spark.read.parquet(str(refined_output_path))


timings = {}
silver_df = spark.read.parquet(str(silver_output_path))

start = perf_counter()
if LOAD_EXISTING_REFINED and refined_output_path.exists():
    loaded_refined_df = spark.read.parquet(str(refined_output_path))
    if loaded_refined_output_matches_parser(loaded_refined_df, parser_probe.model_version):
        refined_df = loaded_refined_df
        refinement_source = "loaded_existing_refined_parquet"
    else:
        refined_df = run_refinement()
        refinement_source = f"reran_{RUN_TIER}_because_existing_parser_output_was_stale"
else:
    refined_df = run_refinement()
    refinement_source = f"ran_{RUN_TIER}"
timings["load_or_run_refinement_seconds"] = perf_counter() - start

start = perf_counter()
write_phase_a_csv_outputs(refined_df)
timings["write_phase_a_csv_outputs_seconds"] = perf_counter() - start

silver_count = silver_df.count()
refined_count = refined_df.count()

refinement_source, silver_count, refined_count, timings

## Before And After Vehicle Rankings

The left table uses the MVP right-context aggregation key, `vehicle_normalized`. The right table uses the refined Vehicle Head lemma among broader Chartable Vehicles.

In [ ]:
mvp_top_vehicles_pdf = top_vehicles_global_dataframe(silver_df).limit(20).toPandas()
chartable_vehicle_heads_pdf = top_chartable_vehicle_heads_dataframe(refined_df).limit(20).toPandas()
clean_common_noun_heads_pdf = top_clean_common_noun_vehicle_heads_dataframe(refined_df).limit(20).toPandas()

display(Markdown("### MVP `vehicle_normalized`"))
display(mvp_top_vehicles_pdf)
display(Markdown("### Refined Chartable Vehicle Head lemma"))
display(chartable_vehicle_heads_pdf)
display(Markdown("### Conservative Clean Common-Noun Vehicle Head lemma"))
display(clean_common_noun_heads_pdf)

In [ ]:
def plot_ranking_or_empty(dataframe, *, x, y, ax, title, xlabel, ylabel, color):
    if dataframe.empty:
        ax.text(
            0.5,
            0.5,
            "No rows available for this ranking",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        dataframe.sort_values(y).plot.barh(
            x=x,
            y=y,
            ax=ax,
            legend=False,
            color=color,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mvp_chart_pdf = mvp_top_vehicles_pdf.head(10)
plot_ranking_or_empty(
    mvp_chart_pdf,
    x="vehicle_normalized",
    y="occurrence_count",
    ax=axes[0],
    title="MVP right-context vehicles",
    xlabel="occurrences",
    ylabel="vehicle_normalized",
    color="#4f6f52",
)

refined_chart_pdf = chartable_vehicle_heads_pdf.head(10)
plot_ranking_or_empty(
    refined_chart_pdf,
    x="vehicle_head_lemma",
    y="occurrence_count",
    ax=axes[1],
    title="Refined Vehicle Head lemmas",
    xlabel="occurrences",
    ylabel="vehicle_head_lemma",
    color="#7a5a8b",
)

plt.tight_layout()

## Scope And Structural Quality Counts

In [ ]:
scope_counts_pdf = refinement_scope_counts_dataframe(refined_df).toPandas()
quality_bucket_counts_pdf = structural_quality_bucket_counts_dataframe(refined_df).toPandas()

display(Markdown("### Counts by `nlp_refinement_scope`"))
display(scope_counts_pdf)
display(Markdown("### Counts by Structural Quality Bucket"))
display(quality_bucket_counts_pdf)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scope_counts_pdf.sort_values("count").plot.barh(
    x="nlp_refinement_scope",
    y="count",
    ax=axes[0],
    legend=False,
    color="#3f6f8f",
)
axes[0].set_title("Refinement scope")
axes[0].set_xlabel("rows")
axes[0].set_ylabel("")

quality_bucket_counts_pdf.sort_values("count").plot.barh(
    x="structural_quality_bucket",
    y="count",
    ax=axes[1],
    legend=False,
    color="#9a6b36",
)
axes[1].set_title("Structural Quality Bucket")
axes[1].set_xlabel("rows")
axes[1].set_ylabel("")

plt.tight_layout()

## Refined Vehicle Heads By Pattern

In [ ]:
vehicle_heads_by_pattern_pdf = top_refined_vehicle_heads_by_pattern_dataframe(refined_df).limit(50).toPandas()
vehicle_heads_by_pattern_pdf

## Side-By-Side Refinement Examples

These examples keep the candidate text and MVP vehicle fields beside the refined Vehicle Phrase, Vehicle Head lemma, head POS, Structural Quality Bucket, and reject reason.

In [ ]:
examples_pdf = (
    refinement_examples_dataframe(refined_df, examples_per_pattern_bucket=5)
    .select(
        "candidate_full_text",
        "pattern_type",
        "vehicle_raw",
        "vehicle_normalized",
        "vehicle_phrase_nlp",
        "vehicle_head_lemma",
        "vehicle_head_pos",
        "structural_quality_bucket",
        "vehicle_reject_reason",
    )
    .limit(60)
    .toPandas()
)

examples_pdf

## Durable Outputs

In [ ]:
display(Markdown(
    "\n".join([
        f"- Refinement source: `{refinement_source}`",
        f"- Requested tier: `{RUN_TIER}`",
        f"- Silver candidate rows: `{silver_count:,}`",
        f"- Refined candidate rows loaded for inspection: `{refined_count:,}`",
        f"- Refined Parquet output inspected: `{refined_output_path.relative_to(REPO_ROOT)}`",
        f"- Canonical full-run Parquet output: `{REFINED_CANDIDATES_NLP_OUTPUT_PATH}`",
        f"- Phase A parser model: `{phase_a_parser_model}`",
        f"- Phase A CSV outputs: `{phase_a_csv_paths}`",
        f"- Timings, seconds: `{ {key: round(value, 2) for key, value in timings.items()} }`",
        "- Interpretation note: Phase A separates structural vehicle quality from semantic judgment.",
        "- Limitation: these outputs do not decide whether a candidate is figurative or literal.",
    ])
))